# 📻 Radio Player — Сборка APK

Этот ноутбук собирает APK для Android-приложения Radio Player.

**Время сборки:** ~15-30 минут (зависит от Colab).

---

## Шаг 1. Установка Buildozer и зависимостей

Нажми `Shift+Enter` на этой ячейке.

In [ ]:
# Установка Buildozer и системных зависимостей
import subprocess, sys

def run(cmd, desc=''):
    print(f'>>> {desc or cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=False)
    if result.returncode != 0:
        print(f'❌ Ошибка при выполнении: {cmd}')
        print(f'   Код возврата: {result.returncode}')
    return result.returncode

# 1. Обновление пакетов
run('apt update -qq', 'Обновление списка пакетов...')

# 2. Установка системных зависимостей
run('apt install -y python3-pip git zip unzip openjdk-17-jdk libltdl-dev libffi-dev libssl-dev libtool libtool-bin autoconf automake', 'Установка системных зависимостей...')

# 3. Установка Buildozer и Cython
# Фиксируем версию buildozer 1.5.0 (совместима с p4a v2024.01.21)
# Более новые версии buildozer могут требовать другой формат buildozer.spec
run('pip install buildozer==1.5.0 cython', 'Установка Buildozer 1.5.0 и Cython...')

# 4. Проверка установки
print()
print('=' * 40)
print('ПРОВЕРКА УСТАНОВКИ:')
print('=' * 40)

# Проверка buildozer
result = subprocess.run('which buildozer || whereis buildozer', shell=True, capture_output=True, text=True)
if result.stdout.strip():
    print(f'✅ Buildozer: {result.stdout.strip()}')
else:
    print('❌ Buildozer НЕ НАЙДЕН! Попробуйте перезапустить сессию Colab (Runtime -> Restart runtime)')

# Проверка Java
result = subprocess.run('java -version 2>&1', shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ Java: {result.stderr.strip().split(chr(10))[0]}')
else:
    print('❌ Java не установлена')

# Проверка Python
result = subprocess.run('python3 --version', shell=True, capture_output=True, text=True)
print(f'✅ {result.stdout.strip()}')

print()
print('✅ Шаг 1 завершён! Можно переходить к Шагу 2.')

## Шаг 2. Загрузка проекта

Нажмите на кнопку **Choose Files** ниже и выберите ZIP-архив `radio_player_android.zip`.

In [ ]:
import zipfile
import os
from google.colab import files

# Функция поиска buildozer.spec
def find_buildozer_spec():
    for root, dirs, files in os.walk('.'):
        if 'buildozer.spec' in files:
            return root
    return None

# Проверяем, есть ли уже файлы проекта
BUILD_DIR = find_buildozer_spec()

if BUILD_DIR:
    print(f"✅ buildozer.spec найден в: {BUILD_DIR}")
else:
    print("📁 Загрузите ZIP-архив с проектом...")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zf:
                zf.extractall()
            os.remove(fn)
            print(f"✅ Распакован: {fn}")

    # Ищем снова после распаковки
    BUILD_DIR = find_buildozer_spec()

if BUILD_DIR:
    print(f"📂 Рабочая папка: {BUILD_DIR}")
    print(f"   Содержимое: {os.listdir(BUILD_DIR)}")
else:
    print("❌ buildozer.spec не найден!")
    print("   Содержимое текущей папки:", os.listdir('.'))

## Шаг 2b. Выбор версии Android

По умолчанию сборка идёт под **Android 5.0+ (API 21–33)** — максимальная совместимость.

Если нужно собрать под конкретную версию — раскомментируй нужную строку ниже и нажми Shift+Enter.

In [ ]:
import os
import re

SPEC_FILE = os.path.join(BUILD_DIR, 'buildozer.spec')

# =============================================
# ВЫБЕРИ ОДИН ВАРИАНТ (убери # в начале строк):
# =============================================

# --- Вариант 1: Android 5.0+ (API 21-33) ---
#ANDROID_API = 33
#ANDROID_MINAPI = 21
#ANDROID_SDK = 33
#ANDROID_NDK = '25b'

# --- Вариант 2: Android 7.0+ (API 24-34) ---
#ANDROID_API = 34
#ANDROID_MINAPI = 24
#ANDROID_SDK = 34
#ANDROID_NDK = '27b'

# --- Вариант 3: Android 15+ (API 35) ---
#ANDROID_API = 35
#ANDROID_MINAPI = 24
#ANDROID_SDK = 35
#ANDROID_NDK = '27b'

# =============================================
# ПРИМЕНЕНИЕ НАСТРОЕК (не редактировать)
# =============================================

if 'ANDROID_API' in dir():
    with open(SPEC_FILE, 'r') as f:
        content = f.read()
    content = re.sub(r'android\.api = \d+', f'android.api = {ANDROID_API}', content)
    content = re.sub(r'android\.minapi = \d+', f'android.minapi = {ANDROID_MINAPI}', content)
    content = re.sub(r'android\.sdk = \d+', f'android.sdk = {ANDROID_SDK}', content)
    content = re.sub(r'android\.ndk = \w+', f'android.ndk = {ANDROID_NDK}', content)
    with open(SPEC_FILE, 'w') as f:
        f.write(content)
    print(f'✅ Настройки обновлены: API {ANDROID_API}, minAPI {ANDROID_MINAPI}, SDK {ANDROID_SDK}, NDK {ANDROID_NDK}')
else:
    print('ℹ️ Используются настройки по умолчанию (Android 5.0+, API 21-33)')

# Покажем текущие настройки
print()
print('📋 ТЕКУЩИЕ НАСТРОЙКИ buildozer.spec:')
print('=' * 40)
with open(SPEC_FILE, 'r') as f:
    for line in f:
        line = line.rstrip()
        if any(line.startswith(k) for k in ['android.api', 'android.minapi', 'android.sdk', 'android.ndk', 'p4a.branch', 'p4a.setup_py', 'requirements']):
            print(f'   {line}')
print('=' * 40)

## Шаг 3. Сборка APK

**Внимание:** сборка занимает 15-30 минут. Colab может отключиться при бездействии — периодически проверяйте процесс.

Нажмите `Shift+Enter` и ждите.

In [ ]:
import os
import re

if BUILD_DIR is None:
    raise RuntimeError("buildozer.spec не найден. Выполните Шаг 2 сначала.")

SPEC_FILE = os.path.join(BUILD_DIR, 'buildozer.spec')

# Диагностика: проверяем, что version задана в buildozer.spec
print("🔍 Диагностика buildozer.spec...")
with open(SPEC_FILE, 'r') as f:
    content = f.read()

# Проверяем наличие version в секции [app]
app_section = re.search(r'\[app\](.*?)(?:\[|$)', content, re.DOTALL)
if app_section:
    app_text = app_section.group(1)
    has_version = 'version' in [line.strip().split('=')[0].strip() for line in app_text.split('\n') if '=' in line]
    print(f"   📋 Секция [app] найдена")
    print(f"   📋 version задана: {has_version}")
    if not has_version:
        print("   ⚠️ version не найдена! Добавляем принудительно...")
        # Вставляем version=1 после [app]
        content = content.replace('[app]', '[app]\nversion = 1', 1)
        with open(SPEC_FILE, 'w') as f:
            f.write(content)
        print("   ✅ version=1 добавлена")
else:
    print("   ❌ Секция [app] не найдена!")

# Покажем финальное содержимое секции [app]
print()
print("📋 Секция [app] в buildozer.spec:")
print('=' * 40)
with open(SPEC_FILE, 'r') as f:
    in_app = False
    for line in f:
        line = line.rstrip()
        if line.startswith('[app]'):
            in_app = True
        elif line.startswith('[') and in_app:
            break
        if in_app:
            print(f'   {line}')
print('=' * 40)
print()

print("🔨 Начинаем сборку APK...")
print(f"   Рабочая папка: {BUILD_DIR}")
print()

# Buildozer может запрашивать подтверждение — отвечаем 'yes'
!cd "{BUILD_DIR}" && yes | buildozer -v android debug 2>&1

print()
print("✅ Сборка завершена!")

## Шаг 4. Скачать APK

После успешной сборки APK скачается автоматически.

In [ ]:
from google.colab import files
import glob

# Ищем APK в bin/ относительно рабочей папки
apk_dir = os.path.join(BUILD_DIR, 'bin')
apk_files = glob.glob(os.path.join(apk_dir, '*.apk'))

if apk_files:
    print(f"📦 Найдено APK: {len(apk_files)}")
    for apk in apk_files:
        size_mb = os.path.getsize(apk) / (1024 * 1024)
        print(f"   📄 {os.path.basename(apk)} ({size_mb:.1f} MB)")
        files.download(apk)
    print("✅ APK скачивается!")
else:
    print("❌ APK не найден в папке bin/")
    print()
    print("📋 Диагностика:")
    for d in [apk_dir, BUILD_DIR]:
        if os.path.exists(d):
            items = os.listdir(d)
            print(f"   Содержимое {d}/: {items}")
            for item in items:
                item_path = os.path.join(d, item)
                if os.path.isdir(item_path):
                    print(f"      {item}/: {os.listdir(item_path)}")

---

## 🔧 Устранение проблем

### 1. Colab отключился во время сборки
Запустите ноутбук заново. На Шаге 2 файлы уже будут на месте (Colab сохраняет их в сессии).

### 2. Ошибка "java not found"
Выполните Шаг 1 ещё раз — возможно, не установилась Java.

### 3. APK не скачивается
После Шага 3 откройте папку `bin/` в боковой панели Colab (📁) и скачайте APK вручную.

### 4. Хочу собрать под другую версию Android
Используйте **Шаг 2b** перед сборкой — там можно выбрать API 33 (5.0+), 34 (7.0+) или 35 (15+).

### 5. Ошибка компиляции Kivy ("_PyLong_AsByteArray" / Python 3.14)
Если сборка падает с ошибками вроде `too few arguments to function call, expected 6, have 5`
в файлах `kivy/graphics/*.c` — это значит, что python-for-android скачал Python 3.14,
который несовместим с Kivy 2.3.0.

**Решение:** В `buildozer.spec` уже прописаны:
- `p4a.branch = v2024.01.21` (использует Python 3.11.5)
- `python3==3.11.5` в requirements

Если вы вручную меняли эти настройки — верните их обратно.

---

## 📱 Установка APK

1. Перенесите APK на телефон (через USB, Telegram, Google Drive и т.д.)
2. На телефоне откройте файл APK
3. Подтвердите установку из неизвестных источников
4. Готово! 🎉